# M3 (Milind) — Priority *double-check*

**Milind's Option B.** Do not try to predict the exact priority — the labels are
inconsistent, so no model does better than ~50%. Instead estimate the priority
and **flag** cases where the agent's priority is `FLAG_GAP` (=2) or more levels
away. Inbound features only; **`sla_target_hours` is excluded** (it is derived
from priority). Chronological split. Saves `models/m3_priority_milind.joblib`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import northstar_prep as P
from sklearn.metrics import f1_score, accuracy_score

df = P.load_clean()
FEATS, TARGET, FLAG_GAP = ["case_summary","case_type","channel"], "priority", 2
tr, va, te = P.time_split(df)
pipe = P.make_clf([P.t_word(), P.t_cat(["case_type","channel"])]).fit(tr[FEATS], tr[TARGET])
pred = pipe.predict(te[FEATS])
rank = P.PRIORITY_RANK
gap  = (te[TARGET].map(rank) - [rank[p] for p in pred]).abs()
metrics = {"macro_F1": round(f1_score(te[TARGET],pred,average="macro"),3),
           "accuracy": round(accuracy_score(te[TARGET],pred),3),
           "within_1_level": round((gap<=1).mean(),3),
           "flag_rate_vs_recorded": round((gap>=FLAG_GAP).mean(),3),
           "baseline_acc": round(df[TARGET].value_counts(normalize=True).iloc[0],3)}
print(metrics)

In [ ]:
proba = pipe.predict_proba(te[FEATS])
test = te[["case_id","case_summary","case_type","channel",TARGET]].copy().reset_index(drop=True)
test["predicted"], test["confidence"] = pred, proba.max(axis=1).round(3)
full = P.make_clf([P.t_word(), P.t_cat(["case_type","channel"])]).fit(df[FEATS], df[TARGET])
path = P.save_bundle("m3_priority_milind", {
  "model": full, "target": TARGET, "features": FEATS, "classes": P.PRIORITIES,
  "kind":"multiclass", "label":"Priority (double-check)", "metrics": metrics, "test": test,
  "meta":{"author":"Milind","approach":"double-check (flag >=2-level mismatch)","split":"time",
          "flag_gap":FLAG_GAP,"priorities":P.PRIORITIES,"inputs":"summary + case_type + channel (no sla_target, no plan proxy)"}})
print("saved", path)